In [1]:
import pandas as pd
import numpy as np
import re

# Load the datasets
train_df = pd.read_csv("/kaggle/input/price-dataset/train.csv")
test_df = pd.read_csv("/kaggle/input/price-dataset/test.csv")

# Apply log transformation to the target variable
train_df['price'] = np.log1p(train_df['price'])

In [2]:
def extract_ipq(text):
    # Define a list of regex patterns to find IPQ
    # This list can be expanded as you explore the data
    patterns = [
        r"(pack of|pk of|pack)\s*(\d+)",
        r"(\d+)\s*count",
        r"(\d+)\s*pk",
        r"ipq\s*:\s*(\d+)"
    ]
    
    text = text.lower()
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Return the first captured number
            for group in match.groups():
                if group.isdigit():
                    return int(group)
    
    # If no pattern matches, assume a quantity of 1
    return 1

# Apply the function to both training and test data
train_df['item_quantity'] = train_df['catalog_content'].apply(extract_ipq)
test_df['item_quantity'] = test_df['catalog_content'].apply(extract_ipq)

In [3]:
# Save the processed dataframes
train_df.to_csv("/kaggle/working/train_processed.csv", index=False)
test_df.to_csv("/kaggle/working/test_processed.csv", index=False)

print("Processed data saved.")

Phase 1 complete. Processed data saved.


In [4]:
import pandas as pd
import numpy as np

# Load data from Phase 1
train_df = pd.read_csv("/kaggle/input/price-dataset/train.csv")
test_df = pd.read_csv("/kaggle/input/price-dataset/test.csv")

In [5]:
!pip install --upgrade transformers huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour

In [6]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm
import numpy as np

# Ensure you are using the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased').to(device)

def get_bert_embeddings(texts, batch_size=32):
    model.eval()
    all_embeddings =[]  # This line is corrected
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        # Use the token's embedding as the sentence representation
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
    return np.vstack(all_embeddings)

# Generate and save embeddings
# Make sure train_df and test_df are loaded before running this
train_text_embeddings = get_bert_embeddings(train_df['catalog_content'].tolist())
np.save('/kaggle/working/train_text_embeddings.npy', train_text_embeddings)

test_text_embeddings = get_bert_embeddings(test_df['catalog_content'].tolist())
np.save('/kaggle/working/test_text_embeddings.npy', test_text_embeddings)

2025-10-13 06:43:57.029121: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760337837.174493      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760337837.216509      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

100%|██████████| 2344/2344 [07:16<00:00,  5.36it/s]


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))

train_tfidf = tfidf.fit_transform(train_df['catalog_content'])
save_npz('/kaggle/working/train_tfidf.npz', train_tfidf)

test_tfidf = tfidf.transform(test_df['catalog_content'])
save_npz('/kaggle/working/test_tfidf.npz', test_tfidf)

print("All features extracted and saved.")

Phase 2 complete. All features extracted and saved.


In [ ]:
# ==================================================
#Splitting the data for Parallel Processing 
# ==================================================

# ==================================================
# CHANGE THIS VALUE IN EACH NOTEBOOK (1, 2, 3, or 4)
PART_NUMBER = 1
# ==================================================

import pandas as pd
import requests
from tqdm import tqdm
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Define Paths ---
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/KagglePricingProject/'
TRAIN_CSV_PATH = os.path.join(DRIVE_PROJECT_PATH, f'train_part_{PART_NUMBER}.csv')
TEST_CSV_PATH = os.path.join(DRIVE_PROJECT_PATH, f'test_part_{PART_NUMBER}.csv')
TRAIN_IMAGES_DIR = os.path.join(DRIVE_PROJECT_PATH, f'train_images_part_{PART_NUMBER}/')
TEST_IMAGES_DIR = os.path.join(DRIVE_PROJECT_PATH, f'test_images_part_{PART_NUMBER}/')

os.makedirs(TRAIN_IMAGES_DIR, exist_ok=True)
os.makedirs(TEST_IMAGES_DIR, exist_ok=True)

# --- Load Data ---
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

# --- Downloader Function ---
def download_images(df, image_directory):
    for index, row in tqdm(df.iterrows(), total=len(df), desc=f"Part {PART_NUMBER}"):
        output_path = os.path.join(image_directory, f"{row['sample_id']}.jpg")
        if os.path.exists(output_path):
            continue
        try:
            response = requests.get(row['image_link'], timeout=15)
            response.raise_for_status()
            with open(output_path, 'wb') as f:
                f.write(response.content)
        except requests.exceptions.RequestException as e:
            print(f"\nCould not download for sample_id {row['sample_id']}. Error: {e}")

# --- Run Downloads ---
print(f"--- Downloading training images for Part {PART_NUMBER} ---")
download_images(train_df, TRAIN_IMAGES_DIR)
print(f"--- Downloading test images for Part {PART_NUMBER} ---")
download_images(test_df, TEST_IMAGES_DIR)
print(f"\nAll downloads for Part {PART_NUMBER} are complete.")

In [ ]:
# ==================================================
# PARALLEL EMBEDDEDING 
PART_NUMBER = 1
# ==================================================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet import preprocess_input
from PIL import Image
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Define Paths ---
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/KagglePricingProject/'
TRAIN_CSV_PATH = os.path.join(DRIVE_PROJECT_PATH, f'train_part_{PART_NUMBER}.csv')
TEST_CSV_PATH = os.path.join(DRIVE_PROJECT_PATH, f'test_part_{PART_NUMBER}.csv')
TRAIN_IMAGES_DIR = os.path.join(DRIVE_PROJECT_PATH, f'train_images_part_{PART_NUMBER}/')
TEST_IMAGES_DIR = os.path.join(DRIVE_PROJECT_PATH, f'test_images_part_{PART_NUMBER}/')
TRAIN_EMBEDDINGS_OUTPUT_PATH = os.path.join(DRIVE_PROJECT_PATH, f'train_image_embeddings_part_{PART_NUMBER}.npy')
TEST_EMBEDDINGS_OUTPUT_PATH = os.path.join(DRIVE_PROJECT_PATH, f'test_image_embeddings_part_{PART_NUMBER}.npy')

# --- Load Data ---
train_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

# --- Set up Model and Functions ---
base_model = EfficientNetB0(weights='imagenet', include_top=False, pooling='avg')
BATCH_SIZE = 64

def local_image_generator(df, image_directory):
    for index, row in df.iterrows():
        image_path = os.path.join(image_directory, f"{row['sample_id']}.jpg")
        try:
            img = Image.open(image_path).convert('RGB')
            img = img.resize((224, 224))
            img_array = image.img_to_array(img)
            img_array = preprocess_input(img_array)
            yield img_array
        except (FileNotFoundError, IOError):
            yield np.zeros((224, 224, 3))

def get_all_image_embeddings(df, image_directory):
    dataset = tf.data.Dataset.from_generator(
        lambda: local_image_generator(df, image_directory),
        output_signature=tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32)
    ).batch(BATCH_SIZE)
    return base_model.predict(dataset, verbose=1)

# --- Run Embedding Generation ---
print(f"--- Generating training embeddings for Part {PART_NUMBER} ---")
train_embeddings = get_all_image_embeddings(train_df, TRAIN_IMAGES_DIR)
np.save(TRAIN_EMBEDDINGS_OUTPUT_PATH, train_embeddings)
print(f"Saved {TRAIN_EMBEDDINGS_OUTPUT_PATH}")

print(f"--- Generating test embeddings for Part {PART_NUMBER} ---")
test_embeddings = get_all_image_embeddings(test_df, TEST_IMAGES_DIR)
np.save(TEST_EMBEDDINGS_OUTPUT_PATH, test_embeddings)
print(f"Saved {TEST_EMBEDDINGS_OUTPUT_PATH}")

In [ ]:
# ==================================================
#Combining the data 
# ==================================================


import numpy as np
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Define Paths ---
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/KagglePricingProject/'
NUM_PARTS = 4

# --- Combine Training Embeddings ---
print("Combining training embedding parts...")
train_parts =
for i in range(NUM_PARTS):
    part_path = os.path.join(DRIVE_PROJECT_PATH, f'train_image_embeddings_part_{i+1}.npy')
    print(f"Loading {part_path}")
    train_parts.append(np.load(part_path))

# Concatenate arrays vertically in the correct order
full_train_embeddings = np.vstack(train_parts)
final_train_path = os.path.join(DRIVE_PROJECT_PATH, 'train_image_embeddings.npy')
np.save(final_train_path, full_train_embeddings)
print(f"Successfully saved combined training embeddings to {final_train_path} with shape {full_train_embeddings.shape}")

# --- Combine Test Embeddings ---
print("\nCombining test embedding parts...")
test_parts =
for i in range(NUM_PARTS):
    part_path = os.path.join(DRIVE_PROJECT_PATH, f'test_image_embeddings_part_{i+1}.npy')
    print(f"Loading {part_path}")
    test_parts.append(np.load(part_path))

full_test_embeddings = np.vstack(test_parts)
final_test_path = os.path.join(DRIVE_PROJECT_PATH, 'test_image_embeddings.npy')
np.save(final_test_path, full_test_embeddings)
print(f"Successfully saved combined test embeddings to {final_test_path} with shape {full_test_embeddings.shape}")